In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

In [7]:
# function to calculate NCAR
def DayXReturn(dayX, market_data, ticker, eventDate):

    # enforce dayX value btw 1 and 20
    if not isinstance(dayX, (int, np.integer)) or not (1 <= dayX <= 20):
        raise ValueError ("dayX must be an integer between 1 and 20.")
    
    # filter by ticker, sort by date
    df = market_data[market_data['Ticker'] == ticker].sort_values('Date').reset_index(drop=True)

    # match event date
    match = df.index[pd.to_datetime(df['Date'], utc = True).dt.strftime('%Y-%m-%d') == pd.to_datetime(eventDate, utc = True).strftime('%Y-%m-%d')]
    if len(match)==0:
        return np.nan
    idx = match[0]

    # check data avlb
    if idx < 90 or (idx+dayX+1) > len(df):
        return np.nan

    # gather data from -90 days prior to +20 days post event
    cols = ['Close', 'Volume', 'NBI']
    pre_data = df.iloc[idx-90 : idx][cols].to_numpy(dtype=float)
    post_data = df.iloc[idx+1 : idx+dayX+1]['Close'].to_numpy(dtype=float)

    # extract event date features for normalization
    base_features = df.iloc[idx][['Close', 'Volume', 'NBI']].to_numpy(dtype=float)
    base_close = base_features[0]

    # ensure data is positive
    if (pre_data == 0).any() or (base_features == 0).any():
        return np.nan

    # sliding window 30d input, dayX outcome (n overlapping samples) with rolling base normalization
    nSamples = 90 -30 - dayX +1
    X = [(pre_data[i : i+30]/pre_data[i+29]).flatten() for i in range (nSamples)]
    y = [(pre_data[i+30 : i+30+dayX, 0]/pre_data[i+29, 0]) for i in range (nSamples)]

    X = np.array(X)
    y = np.array(y)

    # check for invalid values
    if np.isnan(X).any() or np.isinf(X).any() or np.isinf(y).any() or np.isnan(post_data).any():
        return np.nan

    # historical window
    event_window = df.iloc[idx-29 : idx +1][cols].to_numpy(dtype=float)
    norm_event_window = (event_window/base_features).flatten()

    # fit regression
    pred = Ridge(alpha = 10).fit(X,y).predict([norm_event_window])[0]

    # calculate cummulative realized vs expected returns
    realized = (post_data[-1] - base_close) / base_close
    expected = (pred[-1] if dayX > 1 else float(pred)) - 1.0

    # provide abnormal return
    return float(realized - expected)

In [8]:
market_data_df = pd.read_csv("datasets/market_ds.csv")

In [9]:
catalyst_df = pd.read_csv("datasets/catalyst_ds.csv")

In [10]:
for x in range(1,21):
    NCAR_index = f"NCAR{x}"
    catalyst_df[NCAR_index] = [DayXReturn(x, market_data_df, ticker, date) 
                         for ticker, date in zip(catalyst_df['company_ticker'], 
                                                             pd.to_datetime(catalyst_df['date'], utc = True).dt.strftime('%Y-%m-%d'))]

    catalyst_df[NCAR_index] = catalyst_df[NCAR_index].fillna(0.0) 
catalyst_df = catalyst_df.reset_index(drop=True)
 

In [11]:
# add NCAR20 quintiles. Quintile 1 is the highest.
def add_NCAR_quintiles(cohort):
    if (len(cohort) >=5):
        q = pd.qcut(cohort.rank(method = "first"), q=5, labels = [5,4,3,2,1])
        return q
    else: 
        return pd.Series(np.nan, index = cohort.index, dtype = 'float64')
catalyst_df['NCAR20_quintiles'] = catalyst_df.groupby('date')['NCAR20'].transform(add_NCAR_quintiles)

In [12]:
catalyst_df_filtered = catalyst_df[(catalyst_df['date'] >= '2025-01-01') & (catalyst_df['date'] <= '2026-06-30')]

In [13]:

company_features_df = pd.read_csv("datasets/company_ds.csv")

In [14]:
final_df = pd.merge(
    catalyst_df_filtered,
    company_features_df,
    left_on='company_ticker',
    right_on='ticker',
    how = 'left',
).drop(columns=['ticker'])

In [15]:
final_df.to_csv("datasets/final_ds_NCAR.csv", index = False)